# 📊 데이터셋 상태 점검 및 시각화 도구

이 노트북은 프로젝트 폴더 내에 저장된 데이터셋(`dataset.csv` 및 이미지 파일들)이 올바르게 존재하는지 점검하고, 데이터의 분포와 실제 이미지를 시각적으로 확인하기 위해 작성된 **초보자 맞춤형 교육용 노트북**입니다.

### 🔍 데이터셋 구조 정보
- **메타데이터 파일 경로**: `data/data/dataset.csv` (이미지 파일명과 라벨 정보가 포함되어 있음)
- **이미지 폴더 경로**: `data/data` 기준 상대 경로 (실제 이미지 파일들이 위치함)

---

## 🛠️ 1단계: 필수 라이브러리 설치 및 불러오기

데이터를 불러오고 그래프를 그리며 이미지를 다루기 위해 필요한 핵심 라이브러리들을 설치하고 불러옵니다.

In [ ]:
# 가상환경에서 필요한 라이브러리를 설치합니다. (최초 1회 실행 후 주석 처리하셔도 좋습니다)
%pip install pandas matplotlib seaborn pillow

In [ ]:
# 필요한 라이브러리를 임포트합니다.
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

# 한글 글꼴 설정 (Windows 환경 대응)
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False # 마이너스 기호 깨짐 방지

print("✨ 모든 라이브러리가 성공적으로 불러와졌습니다!")

## 📂 2단계: 메타데이터 CSV 파일 불러오기 및 기본 탐색

`dataset.csv` 파일을 읽어서 전체 행의 수, 열(Column)의 정보, 상위 5개의 샘플 데이터를 먼저 확인해 봅니다.

In [ ]:
# 데이터 경로 설정
csv_path = "data/data/dataset.csv"
base_image_dir = "data/data"

# CSV 파일 로드
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print(f"✅ 데이터셋을 성공적으로 불러왔습니다! 전체 데이터 수: {len(df)}개")
else:
    print(f"❌ '{csv_path}' 경로에 파일이 존재하지 않습니다. 경로를 다시 확인해주세요.")

In [ ]:
# 상위 5개의 행을 출력합니다.
df.head()

In [ ]:
# 데이터프레임의 결측치 및 데이터 타입 정보 파악
df.info()

## 📊 3단계: 클래스(Label) 분포 분석

각 라벨(Label)별 데이터가 얼마나 고르게 분포하고 있는지 개수를 세어보고, 멋진 막대그래프(Bar Chart)로 시각화해 봅니다.

In [ ]:
# 클래스별 데이터 개수 확인
label_counts = df['label'].value_counts().sort_index()
print("=== 클래스별 데이터 개수 ===")
print(label_counts)

In [ ]:
# 라벨 분포 막대그래프 시각화
plt.figure(figsize=(8, 5))
sns.barplot(x=label_counts.index, y=label_counts.values, palette='viridis')
plt.title('데이터셋 클래스(Label) 분포 현황', fontsize=15, pad=15)
plt.xlabel('클래스 (Label)', fontsize=12)
plt.ylabel('이미지 개수', fontsize=12)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

## 🖼️ 4단계: 실제 이미지 파일 시각화하기

각 클래스별로 실제 이미지 데이터를 불러와서 눈으로 확인해 봅니다. 이미지가 손상되지 않고 잘 열리는지 확인하는 과정입니다.

In [ ]:
# 각 라벨별로 임의의 샘플 3개씩 시각화해보기
unique_labels = df['label'].unique()
num_samples = 3

fig, axes = plt.subplots(len(unique_labels), num_samples, figsize=(12, 4 * len(unique_labels)))

for i, label in enumerate(sorted(unique_labels)):
    # 해당 라벨의 샘플 데이터 추출
    sample_df = df[df['label'] == label].sample(n=num_samples, random_state=42)
    
    for j in range(num_samples):
        row = sample_df.iloc[j]
        # 경로 병합 (base_image_dir + path)
        img_path = os.path.join(base_image_dir, row['path'])
        
        ax = axes[i, j] if len(unique_labels) > 1 else axes[j]
        
        try:
            # 이미지 불러오기 및 출력
            img = Image.open(img_path)
            ax.imshow(img)
            ax.set_title(f"Label: {label}\n{os.path.basename(img_path)}", fontsize=10)
        except Exception as e:
            ax.text(0.5, 0.5, f"이미지 로드 실패\n{e}", ha='center', va='center', color='red')
            ax.set_title(f"Label: {label} (에러)", color='red')
            
        ax.axis('off')

plt.tight_layout()
plt.suptitle('클래스별 샘플 이미지 시각화', fontsize=16, y=1.02, weight='bold')
plt.show()

## 🏁 5단계: 데이터셋 최종 상태 진단

간단한 유효성 검사 코드를 실행하여, CSV 상의 모든 파일들이 누락 없이 디렉토리에 잘 들어있는지 빠르게 전수 조사합니다.

In [ ]:
# CSV 파일에 적힌 이미지들이 폴더 내에 실제로 존재하는지 전수 확인
missing_files = []
for index, row in df.iterrows():
    full_path = os.path.join(base_image_dir, row['path'])
    if not os.path.exists(full_path):
        missing_files.append(row['path'])

print("=== 🔍 데이터셋 유효성 전수 검사 결과 ===")
if len(missing_files) == 0:
    print("🎉 대단합니다! CSV에 명시된 모든 이미지 파일이 폴더에 완벽히 존재합니다. (결측 데이터 0개)")
else:
    print(f"⚠️ 경고: 총 {len(missing_files)}개의 이미지 파일이 경로상에서 발견되지 않았습니다.")
    print("누락 데이터 예시:", missing_files[:5])